In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from openai import OpenAI
openai_client = OpenAI()

In [12]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [13]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [14]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—usually you can join if enrollment is still open.

A few things to check:
- **Start date:** If the course has already begun, you may still be able to join late.
- **Enrollment deadline:** Some courses allow rolling enrollment; others don’t.
- **Prerequisites:** Make sure you meet any required background.
- **Access to materials:** You may need to catch up on missed content.

If you want, I can help you write a short message to the course organizer asking whether late enrollment is still possible.


In [15]:
prompt = f"""
Your task is to answer the questions from the course participants based on the provided context.capitalize

Use the conrtext to find the relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know".

Question: {question}

Context: {context}
"""

In [16]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still open.


In [25]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [13]:
print(answer)

Usually, yes — if the course is still open and there are seats available, you can often join after it has started.

A good next step is to check:
- whether registration is still open,
- the course start/end dates,
- any prerequisite or approval requirements,
- and whether you’ll miss any early assignments or live sessions.

If you want, I can help you draft a short message to the instructor or support team asking if late enrollment is allowed.


In [17]:
import requests

docs_url ="https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [18]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [19]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'{url_prefix}{course["path"]}'
    course_response = requests.get(course_url)
    course_response.raise_for_status()  # Check if the request was successful
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [48]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [49]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {
        "question": 2.0,
        "section": 0.5,
        "answer": 1.0
    }
    filter_dict = {"course": course}
    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5)

In [50]:
search_results = search(question)

In [51]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [52]:
INSTRUCTIONS = """
Your task is to answer the questions from the course participants based 
on the provided context.
Use the context to find the relevant information and provide accurate answers.
 If the answer is not found
"""

In [53]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append(f'Q: {doc["question"]}')
        lines.append(f'A: {doc["answer"]}')
        lines.append("")  # Add an empty line for better separation between documents
    
    return "\n".join(lines).strip()

In [61]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    print("context:", context)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context)
    return prompt.strip()

In [64]:
prompt = build_prompt(question, search_results)

context: General Course-Related Questions
Qtn: I just discovered the course. Can I still join?
Ans: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Qtn: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Ans: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Qtn: Certificate: Can I follow the course in a self-paced mode and get a certificate?
Ans: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the t

In [65]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Qtn: I just discovered the course. Can I still join?
Ans: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Qtn: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Ans: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Qtn: Certificate: Can I follow the course in a self-paced mode and get a certificate?
Ans: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting 